# Exploratory Analysis

In [3]:
import pandas as pd
from pathlib import Path

In [4]:
data_dir = Path("stock_control_synthetic_data")

In [5]:
tables = {}
for csv_file in data_dir.glob("*.csv"):
    tables[csv_file.stem] = pd.read_csv(csv_file)

# quick sanity check
for name, df in tables.items():
    print(f"{name}: {df.shape}")

business: (1, 5)
category: (6, 3)
customer: (300, 7)
inventory_movement: (99014, 7)
manufacturer: (8, 4)
other_movements: (150, 5)
price_history: (64, 5)
product: (60, 11)
purchase: (144, 9)
purchase_item: (2058, 6)
sale: (32462, 11)
sale_item: (96956, 7)
store: (4, 7)
store_product: (240, 5)
suplier: (10, 8)
user: (12, 8)
user_store: (14, 2)


## Checking stock in all stores 

In [18]:
sale = tables["sale"]
sale_item = tables["sale_item"]
store = tables["store"]
product = tables["product"]
purchase = tables["purchase"]
purchase_item = tables["purchase_item"]

In [19]:
# --- fix mixed date formats ---
def parse_mixed_dates(s):
    parsed = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    mask = parsed.isna()
    parsed[mask] = pd.to_datetime(s[mask], format="%d/%m/%Y %H:%M", errors="coerce")
    return parsed

In [20]:
sale["sale_date"] = parse_mixed_dates(sale["sale_date"])
purchase["purchase_date"] = pd.to_datetime(purchase["purchase_date"])

In [22]:
# --- join sale_item -> sale (store_id, date) -> product (name/category) ---
si = sale_item.merge(sale[["sale_id", "store_id", "sale_date"]], on="sale_id")
si = si.merge(product[["product_id", "name", "category_id"]], on="product_id")
si = si.merge(store[["store_id", "name"]], on="store_id", suffixes=("_product", "_store"))

In [23]:
# =========================================================
# 1. Best-selling products per store (by quantity sold)
# =========================================================
by_qty = (si.groupby(["name_store", "name_product"])["quantity"]
            .sum()
            .reset_index()
            .sort_values(["name_store", "quantity"], ascending=[True, False]))

top5_qty = by_qty.groupby("name_store").head(5)
print(top5_qty)


                     name_store         name_product  quantity
49              Flagship Centro   Sparkling Water 1L      2333
8               Flagship Centro   Coffee Drink 300ml      2327
17              Flagship Centro   Energy Drink 250ml      2301
35              Flagship Centro              Milk 1L      2293
22              Flagship Centro     Floor Cleaner 1L      2261
109             Kiosk San Roque   Sparkling Water 1L      1012
95              Kiosk San Roque              Milk 1L       980
85              Kiosk San Roque          Granola Bar       977
97              Kiosk San Roque          Notebook A4       972
112             Kiosk San Roque              Stapler       957
139              Mall El Bosque   Fabric Softener 1L      1978
169              Mall El Bosque   Sparkling Water 1L      1963
128              Mall El Bosque   Coffee Drink 300ml      1928
121              Mall El Bosque  All-Purpose Cleaner      1918
171              Mall El Bosque   Sports Drink 500ml   

This schema lacks batch-level cost tracking (lot/FIFO tracking), so profit is computed via time-weighted average cost, which average cost only from purchases before the sale date.

In [24]:
# =========================================================
# 2. Time-weighted average cost per product
# =========================================================
# attach purchase_date to each purchase_item
pi = purchase_item.merge(purchase[["purchase_id", "purchase_date"]], on="purchase_id")
pi = pi.sort_values(["product_id", "purchase_date"])

# expanding (cumulative) average cost per product, as of each purchase
pi["cum_avg_cost"] = (
    pi.groupby("product_id")["unit_cost"]
      .expanding()
      .mean()
      .reset_index(level=0, drop=True)
)

# for each sale line, find the most recent cum_avg_cost as of the sale date
# merge_asof requires sorted keys on both sides, matched per product_id
si_sorted = si.sort_values("sale_date")
cost_lookup = pi[["product_id", "purchase_date", "cum_avg_cost"]].sort_values("purchase_date")

si_cost = pd.merge_asof(
    si_sorted,
    cost_lookup,
    left_on="sale_date",
    right_on="purchase_date",
    by="product_id",
    direction="backward",  # only use purchases that happened before the sale
)

# sales that happened before a product's first purchase get no cost match (NaN) —
# fall back to that product's first known cost as a reasonable default
first_cost = pi.groupby("product_id")["unit_cost"].first().rename("first_cost")
si_cost = si_cost.merge(first_cost, on="product_id", how="left")
si_cost["cum_avg_cost"] = si_cost["cum_avg_cost"].fillna(si_cost["first_cost"])

si_cost["cogs"] = si_cost["cum_avg_cost"] * si_cost["quantity"]
si_cost["profit"] = si_cost["subtotal"] - si_cost["cogs"]


In [36]:
# profit by product and store
profit_by_product= (si_cost.groupby(["name_store", "name_product"])
                    .agg(total_profit=("profit", "sum"),
                    total_cogs=("cogs", "sum"),
                    total_sales=("subtotal", "sum"),
                    total_qty=("quantity", "sum"))
                    .reset_index()
                    .sort_values(["name_store", "total_profit"], ascending=[True, True]))

In [37]:
profit_by_product[profit_by_product["name_store"] == "Flagship Centro"]

,name_store,name_product,total_profit,total_cogs,total_sales,total_qty
33,Flagship Centro,Marker Set,418.515894,623.684106,1042.20,1930
52,Flagship Centro,Stapler,504.998182,692.121818,1197.12,2064
32,Flagship Centro,Lemonade 500ml,740.356929,1176.373071,1916.73,2148
10,Flagship Centro,Cola 500ml,777.088821,1203.941179,1981.03,2223
43,Flagship Centro,Pretzels 150g,815.265656,1322.534344,2137.80,2036
13,Flagship Centro,Crackers 150g,916.577465,1473.502535,2390.08,2134
7,Flagship Centro,Chocolate Bar 50g,936.708482,1445.411518,2382.12,2036
37,Flagship Centro,Notebook A4,946.795663,1424.874337,2371.67,1993
25,Flagship Centro,Granola Bar,970.768737,1396.791263,2367.56,2041
49,Flagship Centro,Sparkling Water 1L,1064.727668,1753.022332,2817.75,2333


In [25]:
# =========================================================
# 3. Profit per store
# =========================================================
profit_by_store = (si_cost.groupby("name_store")
                    .agg(revenue=("subtotal", "sum"),
                         cogs=("cogs", "sum"),
                         profit=("profit", "sum"))
                    .assign(margin_pct=lambda d: (d["profit"] / d["revenue"] * 100).round(1))
                    .sort_values("profit", ascending=False))

print(profit_by_store)

# sanity check: how many sale lines still lack a cost?
print("Missing cost after fallback:", si_cost["cum_avg_cost"].isna().sum())

                               revenue           cogs         profit  \
name_store                                                             
Flagship Centro             1288557.15  780496.629152  508060.520848   
Mall El Bosque              1091728.54  659539.229184  432189.310816   
Suburban Branch - Calderon   635066.71  383369.142033  251697.567967   
Kiosk San Roque              536690.77  325644.770357  211045.999643   

                            margin_pct  
name_store                              
Flagship Centro                   39.4  
Mall El Bosque                    39.6  
Suburban Branch - Calderon        39.6  
Kiosk San Roque                   39.3  
Missing cost after fallback: 0
